In [4]:
import pandas as pd


In [5]:
# import dataset
dataset = pd.read_csv('../dataset/cleaned_dataset.csv')
dataset.shape

(51093, 2)

In [6]:
dataset.head()

,processed_text,status
0,oh gosh,Anxiety
1,trouble sleeping confused mind restless heart ...,Anxiety
2,wrong back dear forward doubt stay restless re...,Anxiety
3,shifted focus something else still worried,Anxiety
4,restless restless month boy mean,Anxiety


In [20]:
# column selections for X and y
X = dataset['processed_text'].values
y = dataset['status'].values

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [29]:
print(f'Total training samples {len(X_train)}')
print(f'Total testing samples {len(X_test)}')

Total training samples 40790
Total testing samples 10198


In [23]:
have_missing_values = dataset['processed_text'].isnull().sum()

if have_missing_values != 0 :
    dataset.dropna(inplace=True)

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

# create the transform
vectorizer = TfidfVectorizer()

# vectors
X_train_vector = vectorizer.fit_transform(X_train)
X_test_vector = vectorizer.transform(X_test)

In [ ]:
print(X_train_vector)

In [ ]:
print(X_test_vector)

In [39]:
def print_message():
    print("Traning complete")
    print("-" * 20)

In [34]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)

model.fit(X_train_vector, y_train)
print_message()

Training complete
--------------------


In [35]:
# prediction
y_pred = model.predict(X_test_vector)

In [55]:
# print classification report
from sklearn.metrics import classification_report

def print_classification_report(y_test, y_pred, title=""):
    print(f"Classification Report: {title}")
    print(classification_report(y_test, y_pred, zero_division=0))

In [36]:
print_classification_report(y_test=y_test, y_pred=y_pred)

Classification Report:
                      precision    recall  f1-score   support

             Anxiety       0.91      0.40      0.55       725
             Bipolar       0.92      0.14      0.25       505
          Depression       0.52      0.80      0.63      3082
              Normal       0.78      0.95      0.86      3129
Personality disorder       1.00      0.02      0.03       178
              Stress       0.91      0.04      0.09       468
            Suicidal       0.66      0.40      0.50      2111

            accuracy                           0.65     10198
           macro avg       0.82      0.39      0.42     10198
        weighted avg       0.70      0.65      0.61     10198



## Improvements

### setting class_weight to 'balanced'

In [40]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42, 
    n_jobs=1, 
    class_weight='balanced')

# train model
model.fit(X_train_vector, y_train)
print_message()

Traning complete
--------------------


In [45]:
y_pred = model.predict(X_test_vector)
print_classification_report(y_test, y_pred)

Classification Report:
                      precision    recall  f1-score   support

             Anxiety       0.88      0.47      0.61       725
             Bipolar       0.94      0.28      0.43       505
          Depression       0.54      0.78      0.64      3082
              Normal       0.77      0.95      0.85      3129
Personality disorder       0.86      0.03      0.06       178
              Stress       0.94      0.04      0.07       468
            Suicidal       0.66      0.42      0.51      2111

            accuracy                           0.66     10198
           macro avg       0.80      0.42      0.45     10198
        weighted avg       0.70      0.66      0.63     10198



### Handling data imbalance

#### SMOTE

In [ ]:
! pip install imbalanced-learn

In [47]:
from imblearn.over_sampling import SMOTE

In [49]:
smote = SMOTE(sampling_strategy='auto', random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train_vector, y_train)

print(f"Original train shape: {X_train_vector.shape}")
print(f"Resampled train shape: {X_train_resampled.shape}")

Original train shape: (40790, 60384)
Resampled train shape: (89670, 60384)


In [51]:
# train the model with resampled data
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=1)
model.fit(X_train_resampled, y_train_resampled)
print_message()

Traning complete
--------------------


In [53]:
y_pred = model.predict(X_test_vector)
print_classification_report(y_test=y_test, y_pred=y_pred)

Classification Report:
                      precision    recall  f1-score   support

             Anxiety       0.78      0.68      0.72       725
             Bipolar       0.76      0.58      0.65       505
          Depression       0.63      0.63      0.63      3082
              Normal       0.79      0.95      0.86      3129
Personality disorder       0.73      0.17      0.27       178
              Stress       0.64      0.17      0.27       468
            Suicidal       0.57      0.59      0.58      2111

            accuracy                           0.69     10198
           macro avg       0.70      0.54      0.57     10198
        weighted avg       0.69      0.69      0.68     10198



## Feature engineering and Hyperparameter tuning

### N-grams

In [56]:
biGram_vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=5000
)

triGram_vectorizer = TfidfVectorizer(
    ngram_range=(1,3),
    max_features=5000
)

# vectors
X_train_vector = biGram_vectorizer.fit_transform(X_train)
X_test_vector = biGram_vectorizer.transform(X_test)

# train model
model.fit(X_train_vector, y_train)
print_message()
y_pred = model.predict(X_test_vector)
print_classification_report(y_test, y_pred, title="With bi-gram (base)")

# bi-grams with balanced model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42, 
    n_jobs=1, 
    class_weight='balanced')

# train model
model.fit(X_train_vector, y_train)
print_message()
y_pred = model.predict(X_test_vector)
print_classification_report(y_test, y_pred, title="With bi-gram (balanced)")

# bi-grams with SMOTE
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_vector, y_train)

print(f"Original train shape: {X_train_vector.shape}")
print(f"Resampled train shape: {X_train_resampled.shape}")

# train the model with resampled data
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=1)
model.fit(X_train_resampled, y_train_resampled)
print_message()
y_pred = model.predict(X_test_vector)
print_classification_report(y_test=y_test, y_pred=y_pred, title="With bi-gram (SMOTE)")

Traning complete
--------------------
Classification Report: With bi-gram (base)
                      precision    recall  f1-score   support

             Anxiety       0.83      0.53      0.64       725
             Bipolar       0.87      0.37      0.52       505
          Depression       0.55      0.78      0.64      3082
              Normal       0.82      0.94      0.87      3129
Personality disorder       0.92      0.06      0.12       178
              Stress       0.86      0.07      0.12       468
            Suicidal       0.66      0.47      0.55      2111

            accuracy                           0.68     10198
           macro avg       0.79      0.46      0.50     10198
        weighted avg       0.71      0.68      0.66     10198

Traning complete
--------------------
Classification Report: With bi-gram (balanced)
                      precision    recall  f1-score   support

             Anxiety       0.80      0.62      0.70       725
             Bipolar    

In [57]:
triGram_vectorizer = TfidfVectorizer(
    ngram_range=(1,3),
    max_features=5000
)

# vectors
X_train_vector = triGram_vectorizer.fit_transform(X_train)
X_test_vector = triGram_vectorizer.transform(X_test)

# train model
model.fit(X_train_vector, y_train)
print_message()
y_pred = model.predict(X_test_vector)
print_classification_report(y_test, y_pred, title="With tri-gram (base)")

# bi-grams with balanced model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42, 
    n_jobs=1, 
    class_weight='balanced')

# train model
model.fit(X_train_vector, y_train)
print_message()
y_pred = model.predict(X_test_vector)
print_classification_report(y_test, y_pred, title="With tri-gram (balanced)")

# bi-grams with SMOTE
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_vector, y_train)

print(f"Original train shape: {X_train_vector.shape}")
print(f"Resampled train shape: {X_train_resampled.shape}")

# train the model with resampled data
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=1)
model.fit(X_train_resampled, y_train_resampled)
print_message()
y_pred = model.predict(X_test_vector)
print_classification_report(y_test=y_test, y_pred=y_pred, title="With tri-gram (SMOTE)")

Traning complete
--------------------
Classification Report: With tri-gram (base)
                      precision    recall  f1-score   support

             Anxiety       0.85      0.53      0.65       725
             Bipolar       0.87      0.37      0.52       505
          Depression       0.55      0.79      0.64      3082
              Normal       0.81      0.94      0.87      3129
Personality disorder       0.86      0.03      0.06       178
              Stress       0.83      0.05      0.10       468
            Suicidal       0.66      0.46      0.54      2111

            accuracy                           0.68     10198
           macro avg       0.78      0.45      0.48     10198
        weighted avg       0.71      0.68      0.65     10198

Traning complete
--------------------
Classification Report: With tri-gram (balanced)
                      precision    recall  f1-score   support

             Anxiety       0.82      0.63      0.71       725
             Bipolar  

In [58]:
# train the model with resampled data
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=1)
model.fit(X_train_resampled, y_train_resampled)
print_message()
y_pred = model.predict(X_test_vector)
print_classification_report(y_test=y_test, y_pred=y_pred, title="With tri-gram (SMOTE)")

Traning complete
--------------------
Classification Report: With tri-gram (SMOTE)
                      precision    recall  f1-score   support

             Anxiety       0.73      0.73      0.73       725
             Bipolar       0.75      0.66      0.70       505
          Depression       0.63      0.66      0.65      3082
              Normal       0.83      0.93      0.88      3129
Personality disorder       0.76      0.33      0.46       178
              Stress       0.57      0.26      0.35       468
            Suicidal       0.59      0.57      0.58      2111

            accuracy                           0.71     10198
           macro avg       0.69      0.59      0.62     10198
        weighted avg       0.70      0.71      0.70     10198



In [62]:
net_estimates = [400,800]

for estimate in net_estimates:
    model = RandomForestClassifier(
    n_estimators=estimate,
    random_state=42,
    n_jobs=1)
    model.fit(X_train_resampled, y_train_resampled)
    print_message()
    y_pred = model.predict(X_test_vector)
    print_classification_report(y_test=y_test, y_pred=y_pred, title=f"With tri-gram (SMOTE) with {estimate} estimate")

Traning complete
--------------------
Classification Report: With tri-gram (SMOTE) with 400 estimate
                      precision    recall  f1-score   support

             Anxiety       0.72      0.73      0.73       725
             Bipolar       0.74      0.66      0.70       505
          Depression       0.64      0.67      0.65      3082
              Normal       0.83      0.93      0.88      3129
Personality disorder       0.73      0.33      0.46       178
              Stress       0.60      0.26      0.36       468
            Suicidal       0.60      0.57      0.58      2111

            accuracy                           0.71     10198
           macro avg       0.69      0.59      0.62     10198
        weighted avg       0.70      0.71      0.70     10198

Traning complete
--------------------
Classification Report: With tri-gram (SMOTE) with 800 estimate
                      precision    recall  f1-score   support

             Anxiety       0.73      0.73      0.7